Trong file này chứa:
- **Trong cái này thì hiện tại ban đầu chưa có tiền xử lý**
- Chia tập train và test
- Scale bằng min max
- Smote data
- Chạy và đánh giá với tập test mất cân bằng của mô hình có smote và không có smote
+ Hiện tại đang thấy mô hình base chiến thắng do dữ liệu test nó cũng chỉ nghiên về lớp 1 nhiều nên hầu hết nó đúng.
- Gan cho data(base data) ở đây chỉ gan cho tập train
- Train với mô hình
- Tiếp tục gan cho tập test để cân bằng cho các lớp để xem kết quả
- Kết quả mô hình có gan có tỉ lệ cao hơn mô hình train với dữ liệu cơ bản do
+ Mô hình train với dữ liệu cơ bản nên bị mất cân bằng với các lớp 1(lớp mà chiếm đa số) nên nó sẽ dự đoán sai nhiều với các lớp khác nên tỉ lệ thấp.
+ Mô hình train với dữ liệu Gan do đã có GAN cân bằng rồi nên khả năng dự đoán với các lớp khác sẽ tốt hơn.

In [ ]:
import pandas as pd

file_path = '/content/drive/MyDrive/DATN/fetal_health.csv'
data = pd.read_csv(file_path)
display(data.head())

,baseline value,accelerations,fetal_movement,uterine_contractions,light_decelerations,severe_decelerations,prolongued_decelerations,abnormal_short_term_variability,mean_value_of_short_term_variability,percentage_of_time_with_abnormal_long_term_variability,...,histogram_min,histogram_max,histogram_number_of_peaks,histogram_number_of_zeroes,histogram_mode,histogram_mean,histogram_median,histogram_variance,histogram_tendency,fetal_health
0,120.0,0.000,0.0,0.000,0.000,0.0,0.0,73.0,0.5,43.0,...,62.0,126.0,2.0,0.0,120.0,137.0,121.0,73.0,1.0,2.0
1,132.0,0.006,0.0,0.006,0.003,0.0,0.0,17.0,2.1,0.0,...,68.0,198.0,6.0,1.0,141.0,136.0,140.0,12.0,0.0,1.0
2,133.0,0.003,0.0,0.008,0.003,0.0,0.0,16.0,2.1,0.0,...,68.0,198.0,5.0,1.0,141.0,135.0,138.0,13.0,0.0,1.0
3,134.0,0.003,0.0,0.008,0.003,0.0,0.0,16.0,2.4,0.0,...,53.0,170.0,11.0,0.0,137.0,134.0,137.0,13.0,1.0,1.0
4,132.0,0.007,0.0,0.008,0.000,0.0,0.0,16.0,2.4,0.0,...,53.0,170.0,9.0,0.0,137.0,136.0,138.0,11.0,1.0,1.0


In [ ]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2126 entries, 0 to 2125
Data columns (total 22 columns):
 #   Column                                                  Non-Null Count  Dtype  
---  ------                                                  --------------  -----  
 0   baseline value                                          2126 non-null   float64
 1   accelerations                                           2126 non-null   float64
 2   fetal_movement                                          2126 non-null   float64
 3   uterine_contractions                                    2126 non-null   float64
 4   light_decelerations                                     2126 non-null   float64
 5   severe_decelerations                                    2126 non-null   float64
 6   prolongued_decelerations                                2126 non-null   float64
 7   abnormal_short_term_variability                         2126 non-null   float64
 8   mean_value_of_short_term_variability  

In [ ]:
data.shape

(2126, 22)

In [ ]:
data.duplicated().sum()

np.int64(13)

In [ ]:
data.drop_duplicates(inplace=True)

In [ ]:
data.duplicated().sum()

np.int64(0)

In [ ]:
data.isna().sum().sum()

np.int64(0)

In [ ]:
data.describe().T

,count,mean,std,min,25%,50%,75%,max
baseline value,2113.0,133.304780,9.837451,106.0,126.000,133.000,140.000,160.000
accelerations,2113.0,0.003188,0.003871,0.0,0.000,0.002,0.006,0.019
fetal_movement,2113.0,0.009517,0.046804,0.0,0.000,0.000,0.003,0.481
uterine_contractions,2113.0,0.004387,0.002941,0.0,0.002,0.005,0.007,0.015
light_decelerations,2113.0,0.001901,0.002966,0.0,0.000,0.000,0.003,0.015
severe_decelerations,2113.0,0.000003,0.000057,0.0,0.000,0.000,0.000,0.001
prolongued_decelerations,2113.0,0.000159,0.000592,0.0,0.000,0.000,0.000,0.005
abnormal_short_term_variability,2113.0,46.993848,17.177782,12.0,32.000,49.000,61.000,87.000
mean_value_of_short_term_variability,2113.0,1.335021,0.884368,0.2,0.700,1.200,1.700,7.000
percentage_of_time_with_abnormal_long_term_variability,2113.0,9.795078,18.337073,0.0,0.000,0.000,11.000,91.000


In [ ]:
negative_values_found = False
for column in data.select_dtypes(include=['number']).columns:
    negative_count = (data[column] < 0).sum()
    if negative_count > 0:
        print(f"Cột '{column}' có {negative_count} giá trị nhỏ hơn 0.")
        negative_values_found = True

if not negative_values_found:
    print("Không tìm thấy giá trị nào nhỏ hơn 0 trong các cột số.")

Cột 'histogram_tendency' có 165 giá trị nhỏ hơn 0.


In [ ]:
data["histogram_tendency"].value_counts()

,count
histogram_tendency,
0.0,1110
1.0,838
-1.0,165


In [ ]:
data["fetal_health"].value_counts()

,count
fetal_health,
1.0,1646
2.0,292
3.0,175


In [ ]:
from sklearn.model_selection import train_test_split
X = data.drop('fetal_health', axis=1)
y = data['fetal_health']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Kích thước tập huấn luyện X: {X_train.shape}")
print(f"Kích thước tập kiểm tra X: {X_test.shape}")
print(f"Kích thước tập huấn luyện y: {y_train.shape}")
print(f"Kích thước tập kiểm tra y: {y_test.shape}")

Kích thước tập huấn luyện X: (1690, 21)
Kích thước tập kiểm tra X: (423, 21)
Kích thước tập huấn luyện y: (1690,)
Kích thước tập kiểm tra y: (423,)


Scale data

In [ ]:
from sklearn.preprocessing import MinMaxScaler

# Khởi tạo scaler
scaler = MinMaxScaler()

# Fit trên train, transform cả train và test
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Nếu muốn giữ DataFrame
import pandas as pd
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
print("Đã scale dữ liệu xong")
display(X_train_scaled.head())

Đã scale dữ liệu xong


,baseline value,accelerations,fetal_movement,uterine_contractions,light_decelerations,severe_decelerations,prolongued_decelerations,abnormal_short_term_variability,mean_value_of_short_term_variability,percentage_of_time_with_abnormal_long_term_variability,...,histogram_width,histogram_min,histogram_max,histogram_number_of_peaks,histogram_number_of_zeroes,histogram_mode,histogram_mean,histogram_median,histogram_variance,histogram_tendency
1583,0.481481,0.052632,0.000000,0.333333,0.333333,0.0,0.0,0.240000,0.235294,0.054945,...,0.542373,0.018349,0.250000,0.388889,0.0,0.622047,0.522936,0.550459,0.122047,1.0
1331,0.407407,0.421053,0.020790,0.200000,0.333333,0.0,0.0,0.240000,0.250000,0.000000,...,0.779661,0.064220,0.655172,0.666667,0.0,0.606299,0.495413,0.541284,0.192913,0.5
2083,0.444444,0.000000,0.018711,0.533333,0.000000,0.0,0.0,0.813333,0.279412,0.131868,...,0.135593,0.678899,0.250000,0.111111,0.0,0.535433,0.495413,0.486239,0.003937,0.0
423,0.685185,0.000000,0.000000,0.133333,0.000000,0.0,0.0,0.706667,0.044118,0.000000,...,0.112994,0.724771,0.258621,0.111111,0.1,0.653543,0.642202,0.623853,0.003937,0.5
633,0.703704,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.666667,0.044118,0.054945,...,0.045198,0.899083,0.318966,0.055556,0.0,0.732283,0.743119,0.715596,0.000000,0.5


Đây là mô hình dữ liệu cơ bản và đang dự đoán với dữ liệu lệch ta có thông số đánh giá

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Khởi tạo model
rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    random_state=42,
    n_jobs=-1
)

# Train model với dữ liệu đã scale
rf_model.fit(X_train_scaled, y_train)

# Predict
y_pred = rf_model.predict(X_test_scaled)

# Đánh giá
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))

Accuracy: 0.9527186761229315

Classification Report:
               precision    recall  f1-score   support

         1.0       0.95      1.00      0.97       330
         2.0       0.93      0.72      0.82        58
         3.0       0.97      0.91      0.94        35

    accuracy                           0.95       423
   macro avg       0.95      0.88      0.91       423
weighted avg       0.95      0.95      0.95       423


Confusion Matrix:
 [[329   1   0]
 [ 15  42   1]
 [  1   2  32]]


Smote cho data ban đầu

In [ ]:
from imblearn.over_sampling import SMOTE
from collections import Counter
import pandas as pd

# =============================
# 1. Tách X, y từ toàn bộ data
# =============================
target_col = 'fetal_health'

X_all = data.drop(columns=[target_col]).copy()
y_all = data[target_col].copy().astype(int)

print("Kích thước data gốc:", data.shape)

print("\nPhân bố lớp ban đầu:")
print(Counter(y_all))


# =============================
# 2. SMOTE trên toàn bộ data
# =============================
smote_all = SMOTE(random_state=42)

X_smote_all, y_smote_all = smote_all.fit_resample(X_all, y_all)

# Chuyển lại thành DataFrame / Series
X_smote_all = pd.DataFrame(X_smote_all, columns=X_all.columns)
y_smote_all = pd.Series(y_smote_all, name=target_col)


# =============================
# 3. Gộp lại thành data mới
# =============================
data_smote_all = pd.concat([X_smote_all, y_smote_all], axis=1)

print("\nPhân bố lớp sau SMOTE toàn bộ data:")
print(Counter(y_smote_all))

print("\nKích thước data sau SMOTE toàn bộ:", data_smote_all.shape)

display(data_smote_all.head())

Kích thước data gốc: (2113, 22)

Phân bố lớp ban đầu:
Counter({1: 1646, 2: 292, 3: 175})

Phân bố lớp sau SMOTE toàn bộ data:
Counter({2: 1646, 1: 1646, 3: 1646})

Kích thước data sau SMOTE toàn bộ: (4938, 22)


,baseline value,accelerations,fetal_movement,uterine_contractions,light_decelerations,severe_decelerations,prolongued_decelerations,abnormal_short_term_variability,mean_value_of_short_term_variability,percentage_of_time_with_abnormal_long_term_variability,...,histogram_min,histogram_max,histogram_number_of_peaks,histogram_number_of_zeroes,histogram_mode,histogram_mean,histogram_median,histogram_variance,histogram_tendency,fetal_health
0,120.0,0.000,0.0,0.000,0.000,0.0,0.0,73.0,0.5,43.0,...,62.0,126.0,2.0,0.0,120.0,137.0,121.0,73.0,1.0,2
1,132.0,0.006,0.0,0.006,0.003,0.0,0.0,17.0,2.1,0.0,...,68.0,198.0,6.0,1.0,141.0,136.0,140.0,12.0,0.0,1
2,133.0,0.003,0.0,0.008,0.003,0.0,0.0,16.0,2.1,0.0,...,68.0,198.0,5.0,1.0,141.0,135.0,138.0,13.0,0.0,1
3,134.0,0.003,0.0,0.008,0.003,0.0,0.0,16.0,2.4,0.0,...,53.0,170.0,11.0,0.0,137.0,134.0,137.0,13.0,1.0,1
4,132.0,0.007,0.0,0.008,0.000,0.0,0.0,16.0,2.4,0.0,...,53.0,170.0,9.0,0.0,137.0,136.0,138.0,11.0,1.0,1


Scaler data smote

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from collections import Counter
import pandas as pd

# =============================
# 1. Tách X, y từ data đã SMOTE
# =============================
target_col = 'fetal_health'

X_smote = data_smote_all.drop(columns=[target_col]).copy()
y_smote = data_smote_all[target_col].copy().astype(int)

print("Kích thước data_smote_all:", data_smote_all.shape)

print("\nPhân bố lớp trong data_smote_all:")
print(Counter(y_smote))


# =============================
# 2. Chia train/test từ data SMOTE
# =============================
X_train_smote, X_test_smote, y_train_smote, y_test_smote = train_test_split(
    X_smote,
    y_smote,
    test_size=0.2,
    random_state=42,
    stratify=y_smote
)

print("\nKích thước tập train/test sau khi chia từ data SMOTE:")
print("X_train_smote:", X_train_smote.shape)
print("X_test_smote:", X_test_smote.shape)
print("y_train_smote:", y_train_smote.shape)
print("y_test_smote:", y_test_smote.shape)

print("\nPhân bố y_train_smote:")
print(Counter(y_train_smote))

print("\nPhân bố y_test_smote:")
print(Counter(y_test_smote))


# =============================
# 3. Scale dữ liệu SMOTE
# =============================
scaler_smote = MinMaxScaler()

X_train_smote_scaled = scaler_smote.fit_transform(X_train_smote)
X_test_smote_scaled = scaler_smote.transform(X_test_smote)

# Chuyển lại về DataFrame
X_train_smote_scaled = pd.DataFrame(
    X_train_smote_scaled,
    columns=X_train_smote.columns,
    index=X_train_smote.index
)

X_test_smote_scaled = pd.DataFrame(
    X_test_smote_scaled,
    columns=X_test_smote.columns,
    index=X_test_smote.index
)

print("\nSau khi scale:")
display(X_train_smote_scaled.head())
display(X_test_smote_scaled.head())

Kích thước data_smote_all: (4938, 22)

Phân bố lớp trong data_smote_all:
Counter({2: 1646, 1: 1646, 3: 1646})

Kích thước tập train/test sau khi chia từ data SMOTE:
X_train_smote: (3950, 21)
X_test_smote: (988, 21)
y_train_smote: (3950,)
y_test_smote: (988,)

Phân bố y_train_smote:
Counter({1: 1317, 2: 1317, 3: 1316})

Phân bố y_test_smote:
Counter({3: 330, 1: 329, 2: 329})

Sau khi scale:


,baseline value,accelerations,fetal_movement,uterine_contractions,light_decelerations,severe_decelerations,prolongued_decelerations,abnormal_short_term_variability,mean_value_of_short_term_variability,percentage_of_time_with_abnormal_long_term_variability,...,histogram_width,histogram_min,histogram_max,histogram_number_of_peaks,histogram_number_of_zeroes,histogram_mode,histogram_mean,histogram_median,histogram_variance,histogram_tendency
1251,0.240741,0.210526,0.000000,0.466667,0.066667,0.0,0.0,0.146667,0.191176,0.054945,...,0.411962,0.183486,0.198276,0.055556,0.1,0.519685,0.449541,0.440172,0.033457,1.000000
2069,0.444444,0.000000,0.018711,0.400000,0.000000,0.0,0.0,0.800000,0.352941,0.054945,...,0.137321,0.678899,0.250000,0.055556,0.0,0.551181,0.495413,0.495237,0.003717,0.000000
4320,0.500000,0.000000,0.001568,0.415249,0.183617,0.0,0.4,0.650057,0.334121,0.000000,...,0.583613,0.091743,0.370690,0.235875,0.0,0.529628,0.261702,0.332766,0.269359,0.500000
3151,0.727990,0.031445,0.004347,0.327261,0.000000,0.0,0.0,0.442429,0.108861,0.064793,...,0.331265,0.757892,0.616434,0.272010,0.1,0.733201,0.727628,0.712873,0.009656,0.350637
187,0.592593,0.105263,0.000000,0.400000,0.000000,0.0,0.0,0.413333,0.117647,0.208791,...,0.234590,0.724771,0.439655,0.444444,0.2,0.803150,0.752294,0.752207,0.033457,0.500000


,baseline value,accelerations,fetal_movement,uterine_contractions,light_decelerations,severe_decelerations,prolongued_decelerations,abnormal_short_term_variability,mean_value_of_short_term_variability,percentage_of_time_with_abnormal_long_term_variability,...,histogram_width,histogram_min,histogram_max,histogram_number_of_peaks,histogram_number_of_zeroes,histogram_mode,histogram_mean,histogram_median,histogram_variance,histogram_tendency
4424,0.481481,0.000000,0.000000,0.666803,0.088843,0.0,0.13306,0.257796,0.181332,0.000000,...,0.497788,0.091743,0.241379,0.129592,0.200205,0.307087,0.363996,0.369956,0.192141,0.833675
3586,0.760213,0.000000,0.006237,0.000000,0.000000,0.0,0.00000,0.842888,0.007732,0.747819,...,0.072258,0.793342,0.259509,0.113974,0.000000,0.693725,0.670670,0.652201,0.001763,0.762881
1903,0.629630,0.157895,0.002079,0.466667,0.266667,0.0,0.00000,0.680000,0.147059,0.000000,...,0.371910,0.394495,0.336207,0.222222,0.000000,0.653543,0.577982,0.605367,0.070632,1.000000
4543,0.259259,0.000000,0.000000,0.224234,0.824234,0.0,0.00000,0.626667,0.205882,0.000000,...,0.465020,0.174312,0.269595,0.222222,0.100000,0.244094,0.308592,0.271641,0.222375,0.500000
216,0.388889,0.000000,0.010395,0.000000,0.000000,0.0,0.00000,0.653333,0.058824,0.043956,...,0.492066,0.110092,0.250000,0.222222,0.000000,0.559055,0.513761,0.495237,0.007435,1.000000


Mô hình có smote


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Khởi tạo model
rf_smote_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    random_state=42,
    n_jobs=-1
)

# Train model với dữ liệu đã scale
rf_smote_model.fit(X_train_smote_scaled, y_train_smote)

# Predict
y_pred_smote = rf_smote_model.predict(X_test_smote_scaled)

# Đánh giá
print("Accuracy:", accuracy_score(y_test_smote, y_pred_smote))
print("\nClassification Report:\n", classification_report(y_test_smote, y_pred_smote))
print("\nConfusion Matrix:\n", confusion_matrix(y_test_smote, y_pred_smote))

Accuracy: 0.9817813765182186

Classification Report:
               precision    recall  f1-score   support

           1       0.98      0.98      0.98       329
           2       0.97      0.98      0.97       329
           3       1.00      0.99      1.00       330

    accuracy                           0.98       988
   macro avg       0.98      0.98      0.98       988
weighted avg       0.98      0.98      0.98       988


Confusion Matrix:
 [[321   8   0]
 [  7 321   1]
 [  0   2 328]]


Gan

In [ ]:
!pip install ctgan

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.5/74.5 kB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 11.6 MB/s eta 0:00:00


In [ ]:
import pandas as pd
from ctgan import CTGAN
from sklearn.model_selection import train_test_split

target_col = 'fetal_health'

print("Kích thước data gốc:", data.shape)
display(data.head())

print("\nPhân bố lớp ban đầu:")
print(data[target_col].value_counts().sort_index())


# =============================
# 1. Chuẩn bị data cho GAN
# =============================
data_gan_input = data.copy()

# Đảm bảo target là int
data_gan_input[target_col] = data_gan_input[target_col].astype(int)


# =============================
# 2. Train CTGAN trên toàn bộ data
# =============================
ctgan_model_gan = CTGAN(
    epochs=300,
    batch_size=100,
    verbose=True
)

ctgan_model_gan.fit(
    data_gan_input,
    discrete_columns=[target_col]
)

print("\nĐã train xong CTGAN")


# =============================
# 3. Sinh thêm dữ liệu GAN để cân bằng lớp
# =============================
class_counts_gan = data_gan_input[target_col].value_counts()
max_count_gan = class_counts_gan.max()

synthetic_parts_gan = []

for class_gan, count_gan in class_counts_gan.items():
    need_gan = max_count_gan - count_gan

    if need_gan <= 0:
        continue

    print(f"\nLớp {class_gan} cần sinh thêm {need_gan} mẫu")

    collected_gan = []
    total_collected_gan = 0

    while total_collected_gan < need_gan:
        sample_n_gan = max(500, need_gan * 2)

        fake_batch_gan = ctgan_model_gan.sample(sample_n_gan)

        # Vì target là số, lọc trực tiếp theo số
        fake_class_gan = fake_batch_gan[
            fake_batch_gan[target_col] == class_gan
        ].copy()

        if len(fake_class_gan) > 0:
            collected_gan.append(fake_class_gan)
            total_collected_gan += len(fake_class_gan)

            print(f"Đã lấy được {total_collected_gan}/{need_gan}")

    fake_class_final_gan = pd.concat(
        collected_gan,
        axis=0
    ).iloc[:need_gan].copy()

    synthetic_parts_gan.append(fake_class_final_gan)


# =============================
# 4. Gộp dữ liệu thật + dữ liệu GAN sinh thêm
# =============================
if len(synthetic_parts_gan) > 0:
    data_synthetic_gan = pd.concat(
        synthetic_parts_gan,
        axis=0
    ).reset_index(drop=True)

    data_balanced_gan = pd.concat(
        [data_gan_input, data_synthetic_gan],
        axis=0
    ).reset_index(drop=True)
else:
    data_synthetic_gan = pd.DataFrame(columns=data_gan_input.columns)
    data_balanced_gan = data_gan_input.copy()


print("\nKích thước dữ liệu GAN sinh thêm:")
print(data_synthetic_gan.shape)

print("\nKích thước data sau khi cân bằng bằng GAN:")
print(data_balanced_gan.shape)

print("\nPhân bố lớp sau CTGAN:")
print(data_balanced_gan[target_col].value_counts().sort_index())

display(data_balanced_gan.head())

Kích thước data gốc: (2113, 22)


,baseline value,accelerations,fetal_movement,uterine_contractions,light_decelerations,severe_decelerations,prolongued_decelerations,abnormal_short_term_variability,mean_value_of_short_term_variability,percentage_of_time_with_abnormal_long_term_variability,...,histogram_min,histogram_max,histogram_number_of_peaks,histogram_number_of_zeroes,histogram_mode,histogram_mean,histogram_median,histogram_variance,histogram_tendency,fetal_health
0,120.0,0.000,0.0,0.000,0.000,0.0,0.0,73.0,0.5,43.0,...,62.0,126.0,2.0,0.0,120.0,137.0,121.0,73.0,1.0,2.0
1,132.0,0.006,0.0,0.006,0.003,0.0,0.0,17.0,2.1,0.0,...,68.0,198.0,6.0,1.0,141.0,136.0,140.0,12.0,0.0,1.0
2,133.0,0.003,0.0,0.008,0.003,0.0,0.0,16.0,2.1,0.0,...,68.0,198.0,5.0,1.0,141.0,135.0,138.0,13.0,0.0,1.0
3,134.0,0.003,0.0,0.008,0.003,0.0,0.0,16.0,2.4,0.0,...,53.0,170.0,11.0,0.0,137.0,134.0,137.0,13.0,1.0,1.0
4,132.0,0.007,0.0,0.008,0.000,0.0,0.0,16.0,2.4,0.0,...,53.0,170.0,9.0,0.0,137.0,136.0,138.0,11.0,1.0,1.0



Phân bố lớp ban đầu:
fetal_health
1.0    1646
2.0     292
3.0     175
Name: count, dtype: int64


Gen. (-00.41) | Discrim. (+00.65): 100%|██████████| 300/300 [05:14<00:00,  1.05s/it]



Đã train xong CTGAN

Lớp 2 cần sinh thêm 1354 mẫu
Đã lấy được 830/1354
Đã lấy được 1737/1354

Lớp 3 cần sinh thêm 1471 mẫu
Đã lấy được 830/1471
Đã lấy được 1646/1471

Kích thước dữ liệu GAN sinh thêm:
(2825, 22)

Kích thước data sau khi cân bằng bằng GAN:
(4938, 22)

Phân bố lớp sau CTGAN:
fetal_health
1    1646
2    1646
3    1646
Name: count, dtype: int64


,baseline value,accelerations,fetal_movement,uterine_contractions,light_decelerations,severe_decelerations,prolongued_decelerations,abnormal_short_term_variability,mean_value_of_short_term_variability,percentage_of_time_with_abnormal_long_term_variability,...,histogram_min,histogram_max,histogram_number_of_peaks,histogram_number_of_zeroes,histogram_mode,histogram_mean,histogram_median,histogram_variance,histogram_tendency,fetal_health
0,120.0,0.000,0.0,0.000,0.000,0.0,0.0,73.0,0.5,43.0,...,62.0,126.0,2.0,0.0,120.0,137.0,121.0,73.0,1.0,2
1,132.0,0.006,0.0,0.006,0.003,0.0,0.0,17.0,2.1,0.0,...,68.0,198.0,6.0,1.0,141.0,136.0,140.0,12.0,0.0,1
2,133.0,0.003,0.0,0.008,0.003,0.0,0.0,16.0,2.1,0.0,...,68.0,198.0,5.0,1.0,141.0,135.0,138.0,13.0,0.0,1
3,134.0,0.003,0.0,0.008,0.003,0.0,0.0,16.0,2.4,0.0,...,53.0,170.0,11.0,0.0,137.0,134.0,137.0,13.0,1.0,1
4,132.0,0.007,0.0,0.008,0.000,0.0,0.0,16.0,2.4,0.0,...,53.0,170.0,9.0,0.0,137.0,136.0,138.0,11.0,1.0,1


In [ ]:
# =============================
# 5. Tách X, y từ data đã GAN
# =============================
X_gan = data_balanced_gan.drop(columns=[target_col]).copy()
y_gan = data_balanced_gan[target_col].astype(int)


# =============================
# 6. Chia train/test cho data GAN
# =============================
X_train_gan, X_test_gan, y_train_gan, y_test_gan = train_test_split(
    X_gan,
    y_gan,
    test_size=0.2,
    random_state=42,
    stratify=y_gan
)

print("\nKích thước tập train/test sau GAN:")
print("X_train_gan:", X_train_gan.shape)
print("X_test_gan :", X_test_gan.shape)
print("y_train_gan:", y_train_gan.shape)
print("y_test_gan :", y_test_gan.shape)

print("\nPhân bố y_train_gan:")
print(y_train_gan.value_counts().sort_index())

print("\nPhân bố y_test_gan:")
print(y_test_gan.value_counts().sort_index())


Kích thước tập train/test sau GAN:
X_train_gan: (3950, 21)
X_test_gan : (988, 21)
y_train_gan: (3950,)
y_test_gan : (988,)

Phân bố y_train_gan:
fetal_health
1    1317
2    1317
3    1316
Name: count, dtype: int64

Phân bố y_test_gan:
fetal_health
1    329
2    329
3    330
Name: count, dtype: int64


In [ ]:
from sklearn.preprocessing import MinMaxScaler

# =============================
# 7. Scale dữ liệu GAN
# =============================
scaler_gan = MinMaxScaler()

X_train_gan_scaled = scaler_gan.fit_transform(X_train_gan)
X_test_gan_scaled = scaler_gan.transform(X_test_gan)

X_train_gan_scaled = pd.DataFrame(
    X_train_gan_scaled,
    columns=X_train_gan.columns,
    index=X_train_gan.index
)

X_test_gan_scaled = pd.DataFrame(
    X_test_gan_scaled,
    columns=X_test_gan.columns,
    index=X_test_gan.index
)

display(X_train_gan_scaled.head())
display(X_test_gan_scaled.head())

,baseline value,accelerations,fetal_movement,uterine_contractions,light_decelerations,severe_decelerations,prolongued_decelerations,abnormal_short_term_variability,mean_value_of_short_term_variability,percentage_of_time_with_abnormal_long_term_variability,...,histogram_width,histogram_min,histogram_max,histogram_number_of_peaks,histogram_number_of_zeroes,histogram_mode,histogram_mean,histogram_median,histogram_variance,histogram_tendency
1251,0.301565,0.334697,0.005918,0.576743,0.198719,0.581613,0.300623,0.136507,0.233889,0.074031,...,0.434334,0.264889,0.215349,0.093258,0.103648,0.610762,0.520233,0.516046,0.033343,0.979585
2069,0.488170,0.157282,0.019288,0.523836,0.152524,0.581613,0.300623,0.744582,0.387111,0.074031,...,0.191381,0.693473,0.259805,0.093258,0.004053,0.636286,0.560214,0.563648,0.011469,0.108629
4320,0.606908,0.238104,0.008207,0.356140,0.166312,0.317309,0.599503,0.817088,0.061405,0.792955,...,0.131883,0.628644,0.055393,0.100331,0.003268,0.596014,0.589325,0.559140,0.011829,0.541724
3151,0.464214,0.206380,0.005397,0.472946,0.558248,0.390929,0.400498,0.648439,0.066362,0.020716,...,0.609360,0.559526,0.636119,0.144434,0.104864,0.725220,0.651476,0.748637,0.046357,0.539064
187,0.623882,0.245989,0.005918,0.523836,0.152524,0.581613,0.300623,0.384701,0.164243,0.218378,...,0.277427,0.733157,0.422809,0.466623,0.203243,0.840476,0.784105,0.785791,0.033343,0.544107


,baseline value,accelerations,fetal_movement,uterine_contractions,light_decelerations,severe_decelerations,prolongued_decelerations,abnormal_short_term_variability,mean_value_of_short_term_variability,percentage_of_time_with_abnormal_long_term_variability,...,histogram_width,histogram_min,histogram_max,histogram_number_of_peaks,histogram_number_of_zeroes,histogram_mode,histogram_mean,histogram_median,histogram_variance,histogram_tendency
4424,0.568272,0.177240,0.010980,0.305984,0.238372,0.501960,0.862915,0.687335,0.111617,0.635721,...,0.285477,0.225330,0.233184,0.100167,0.005818,0.586024,0.574636,0.447125,0.020995,-0.000682
3586,0.480733,0.241376,0.005765,0.230171,0.185300,0.238521,0.342060,0.735283,0.042045,0.384665,...,0.164621,0.665447,0.092142,0.087735,0.005761,0.747928,0.596049,0.509982,0.007399,0.541029
1903,0.657810,0.290343,0.007404,0.576743,0.337302,0.581613,0.300623,0.632895,0.192101,0.022479,...,0.398903,0.447434,0.333898,0.253272,0.004053,0.719238,0.632179,0.658852,0.060686,0.979585
4543,0.347639,0.136925,0.002332,0.622316,0.309938,0.651073,0.792149,0.946528,0.100546,0.021964,...,0.587729,0.731262,0.327479,0.255235,0.005725,0.198550,0.559673,0.324503,0.245081,0.523211
216,0.437278,0.157282,0.013346,0.206393,0.152524,0.581613,0.300623,0.608075,0.108526,0.063721,...,0.505195,0.201395,0.259805,0.253272,0.004053,0.642667,0.576206,0.563648,0.014203,0.979585


Mô hình với dữ liệu ctgan

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Khởi tạo model
rf_gan_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    random_state=42,
    n_jobs=-1
)

# Train model với dữ liệu đã scale
rf_gan_model.fit(X_train_gan_scaled, y_train_gan)

# Predict
y_pred_gan = rf_gan_model.predict(X_test_gan_scaled)

# Đánh giá
print("Accuracy:", accuracy_score(y_test_gan, y_pred_gan))
print("\nClassification Report:\n", classification_report(y_test_gan, y_pred_gan))
print("\nConfusion Matrix:\n", confusion_matrix(y_test_gan, y_pred_gan))

Accuracy: 0.9564777327935222

Classification Report:
               precision    recall  f1-score   support

           1       0.97      0.99      0.98       329
           2       0.94      0.93      0.94       329
           3       0.96      0.95      0.95       330

    accuracy                           0.96       988
   macro avg       0.96      0.96      0.96       988
weighted avg       0.96      0.96      0.96       988


Confusion Matrix:
 [[327   2   0]
 [  9 306  14]
 [  1  17 312]]


Next so sánh

Mang smote ra dự đoán với base

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)
import pandas as pd

# =============================
# Hàm đánh giá mô hình
# =============================
def evaluate_model_compare(model_name, y_true, y_pred):
    print("\n" + "="*60)
    print(model_name)
    print("="*60)

    print("Accuracy:", accuracy_score(y_true, y_pred))
    print("Balanced Accuracy:", balanced_accuracy_score(y_true, y_pred))

    print("Precision weighted:", precision_score(y_true, y_pred, average='weighted', zero_division=0))
    print("Recall weighted:", recall_score(y_true, y_pred, average='weighted', zero_division=0))
    print("F1 weighted:", f1_score(y_true, y_pred, average='weighted', zero_division=0))

    print("Precision macro:", precision_score(y_true, y_pred, average='macro', zero_division=0))
    print("Recall macro:", recall_score(y_true, y_pred, average='macro', zero_division=0))
    print("F1 macro:", f1_score(y_true, y_pred, average='macro', zero_division=0))

    print("\nClassification Report:")
    print(classification_report(y_true, y_pred, zero_division=0))

    print("\nConfusion Matrix:")
    print(confusion_matrix(y_true, y_pred))


# =============================
# 1. Base model đánh giá trên test gốc
# =============================
y_pred_base_compare = rf_model.predict(X_test_scaled)

evaluate_model_compare(
    "BASE MODEL - Đánh giá trên tập test gốc",
    y_test,
    y_pred_base_compare
)


# =============================
# 2. SMOTE model đánh giá trên cùng test gốc
# =============================
# Lưu ý:
# Nếu X_test_scaled và X_train_smote_scaled dùng cùng scaler thì dùng trực tiếp.
# Nếu bạn có scaler_smote riêng, nên transform lại X_test bằng scaler_smote.

try:
    X_test_for_smote_compare = scaler_smote.transform(X_test)

    X_test_for_smote_compare = pd.DataFrame(
        X_test_for_smote_compare,
        columns=X_test.columns,
        index=X_test.index
    )

except NameError:
    # Nếu không có scaler_smote thì dùng X_test_scaled hiện tại
    X_test_for_smote_compare = X_test_scaled

y_pred_smote_compare = rf_smote_model.predict(X_test_for_smote_compare)

evaluate_model_compare(
    "SMOTE MODEL - Đánh giá trên cùng tập test gốc",
    y_test,
    y_pred_smote_compare
)


BASE MODEL - Đánh giá trên tập test gốc
Accuracy: 0.9527186761229315
Balanced Accuracy: 0.8784644474299647
Precision weighted: 0.9521711098029322
Recall weighted: 0.9527186761229315
F1 weighted: 0.9501915750195354
Precision macro: 0.9522178304787001
Recall macro: 0.8784644474299647
F1 macro: 0.9105084219951914

Classification Report:
              precision    recall  f1-score   support

         1.0       0.95      1.00      0.97       330
         2.0       0.93      0.72      0.82        58
         3.0       0.97      0.91      0.94        35

    accuracy                           0.95       423
   macro avg       0.95      0.88      0.91       423
weighted avg       0.95      0.95      0.95       423


Confusion Matrix:
[[329   1   0]
 [ 15  42   1]
 [  1   2  32]]

SMOTE MODEL - Đánh giá trên cùng tập test gốc
Accuracy: 0.9787234042553191
Balanced Accuracy: 0.9861720654824103
Precision weighted: 0.9807089045742625
Recall weighted: 0.9787234042553191
F1 weighted: 0.9792147958949

In [ ]:
# =============================
# 3. Bảng so sánh tổng hợp
# =============================
compare_results = pd.DataFrame({
    "Model": ["Base", "SMOTE"],
    "Accuracy": [
        accuracy_score(y_test, y_pred_base_compare),
        accuracy_score(y_test, y_pred_smote_compare)
    ],
    "Balanced Accuracy": [
        balanced_accuracy_score(y_test, y_pred_base_compare),
        balanced_accuracy_score(y_test, y_pred_smote_compare)
    ],
    "Precision Macro": [
        precision_score(y_test, y_pred_base_compare, average='macro', zero_division=0),
        precision_score(y_test, y_pred_smote_compare, average='macro', zero_division=0)
    ],
    "Recall Macro": [
        recall_score(y_test, y_pred_base_compare, average='macro', zero_division=0),
        recall_score(y_test, y_pred_smote_compare, average='macro', zero_division=0)
    ],
    "F1 Macro": [
        f1_score(y_test, y_pred_base_compare, average='macro', zero_division=0),
        f1_score(y_test, y_pred_smote_compare, average='macro', zero_division=0)
    ],
    "Precision Weighted": [
        precision_score(y_test, y_pred_base_compare, average='weighted', zero_division=0),
        precision_score(y_test, y_pred_smote_compare, average='weighted', zero_division=0)
    ],
    "Recall Weighted": [
        recall_score(y_test, y_pred_base_compare, average='weighted', zero_division=0),
        recall_score(y_test, y_pred_smote_compare, average='weighted', zero_division=0)
    ],
    "F1 Weighted": [
        f1_score(y_test, y_pred_base_compare, average='weighted', zero_division=0),
        f1_score(y_test, y_pred_smote_compare, average='weighted', zero_division=0)
    ]
})

display(compare_results)

,Model,Accuracy,Balanced Accuracy,Precision Macro,Recall Macro,F1 Macro,Precision Weighted,Recall Weighted,F1 Weighted
0,Base,0.952719,0.878464,0.952218,0.878464,0.910508,0.952171,0.952719,0.950192
1,SMOTE,0.978723,0.986172,0.957942,0.986172,0.971016,0.980709,0.978723,0.979215


Mang Gan ra dự đoán với base

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)
import pandas as pd

# =============================
# Hàm đánh giá mô hình
# =============================
def evaluate_model_compare(model_name, y_true, y_pred):
    print("\n" + "="*60)
    print(model_name)
    print("="*60)

    print("Accuracy:", accuracy_score(y_true, y_pred))
    print("Balanced Accuracy:", balanced_accuracy_score(y_true, y_pred))

    print("Precision weighted:", precision_score(y_true, y_pred, average='weighted', zero_division=0))
    print("Recall weighted:", recall_score(y_true, y_pred, average='weighted', zero_division=0))
    print("F1 weighted:", f1_score(y_true, y_pred, average='weighted', zero_division=0))

    print("Precision macro:", precision_score(y_true, y_pred, average='macro', zero_division=0))
    print("Recall macro:", recall_score(y_true, y_pred, average='macro', zero_division=0))
    print("F1 macro:", f1_score(y_true, y_pred, average='macro', zero_division=0))

    print("\nClassification Report:")
    print(classification_report(y_true, y_pred, zero_division=0))

    print("\nConfusion Matrix:")
    print(confusion_matrix(y_true, y_pred))


# =============================
# 1. Base model đánh giá trên test gốc
# =============================
y_pred_base_compare_gan = rf_model.predict(X_test_scaled)

evaluate_model_compare(
    "BASE MODEL - Đánh giá trên tập test gốc",
    y_test,
    y_pred_base_compare_gan
)


# =============================
# 2. GAN model đánh giá trên cùng test gốc
# =============================
# Nếu GAN model được train bằng X_train_gan_scaled,
# thì test gốc phải được transform bằng scaler_gan.

X_test_for_gan_compare = scaler_gan.transform(X_test)

X_test_for_gan_compare = pd.DataFrame(
    X_test_for_gan_compare,
    columns=X_test.columns,
    index=X_test.index
)

y_pred_gan_compare = rf_gan_model.predict(X_test_for_gan_compare)

evaluate_model_compare(
    "GAN MODEL - Đánh giá trên cùng tập test gốc",
    y_test,
    y_pred_gan_compare
)


BASE MODEL - Đánh giá trên tập test gốc
Accuracy: 0.9527186761229315
Balanced Accuracy: 0.8784644474299647
Precision weighted: 0.9521711098029322
Recall weighted: 0.9527186761229315
F1 weighted: 0.9501915750195354
Precision macro: 0.9522178304787001
Recall macro: 0.8784644474299647
F1 macro: 0.9105084219951914

Classification Report:
              precision    recall  f1-score   support

         1.0       0.95      1.00      0.97       330
         2.0       0.93      0.72      0.82        58
         3.0       0.97      0.91      0.94        35

    accuracy                           0.95       423
   macro avg       0.95      0.88      0.91       423
weighted avg       0.95      0.95      0.95       423


Confusion Matrix:
[[329   1   0]
 [ 15  42   1]
 [  1   2  32]]

GAN MODEL - Đánh giá trên cùng tập test gốc
Accuracy: 0.9881796690307328
Balanced Accuracy: 0.9769617355824253
Precision weighted: 0.9882926901748211
Recall weighted: 0.9881796690307328
F1 weighted: 0.988224802234854

In [ ]:
# =============================
# 3. Bảng so sánh tổng hợp Base vs GAN
# =============================
compare_base_gan_results = pd.DataFrame({
    "Model": ["Base", "GAN"],
    "Accuracy": [
        accuracy_score(y_test, y_pred_base_compare_gan),
        accuracy_score(y_test, y_pred_gan_compare)
    ],
    "Balanced Accuracy": [
        balanced_accuracy_score(y_test, y_pred_base_compare_gan),
        balanced_accuracy_score(y_test, y_pred_gan_compare)
    ],
    "Precision Macro": [
        precision_score(y_test, y_pred_base_compare_gan, average='macro', zero_division=0),
        precision_score(y_test, y_pred_gan_compare, average='macro', zero_division=0)
    ],
    "Recall Macro": [
        recall_score(y_test, y_pred_base_compare_gan, average='macro', zero_division=0),
        recall_score(y_test, y_pred_gan_compare, average='macro', zero_division=0)
    ],
    "F1 Macro": [
        f1_score(y_test, y_pred_base_compare_gan, average='macro', zero_division=0),
        f1_score(y_test, y_pred_gan_compare, average='macro', zero_division=0)
    ],
    "Precision Weighted": [
        precision_score(y_test, y_pred_base_compare_gan, average='weighted', zero_division=0),
        precision_score(y_test, y_pred_gan_compare, average='weighted', zero_division=0)
    ],
    "Recall Weighted": [
        recall_score(y_test, y_pred_base_compare_gan, average='weighted', zero_division=0),
        recall_score(y_test, y_pred_gan_compare, average='weighted', zero_division=0)
    ],
    "F1 Weighted": [
        f1_score(y_test, y_pred_base_compare_gan, average='weighted', zero_division=0),
        f1_score(y_test, y_pred_gan_compare, average='weighted', zero_division=0)
    ]
})

display(compare_base_gan_results)

,Model,Accuracy,Balanced Accuracy,Precision Macro,Recall Macro,F1 Macro,Precision Weighted,Recall Weighted,F1 Weighted
0,Base,0.952719,0.878464,0.952218,0.878464,0.910508,0.952171,0.952719,0.950192
1,GAN,0.988180,0.976962,0.972514,0.976962,0.974714,0.988293,0.988180,0.988225


Mang base ra dự đoán với SMOTE test

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)
import pandas as pd

# =============================
# Hàm đánh giá mô hình
# =============================
def evaluate_model_compare(model_name, y_true, y_pred):
    print("\n" + "="*60)
    print(model_name)
    print("="*60)

    print("Accuracy:", accuracy_score(y_true, y_pred))
    print("Balanced Accuracy:", balanced_accuracy_score(y_true, y_pred))

    print("Precision weighted:", precision_score(y_true, y_pred, average='weighted', zero_division=0))
    print("Recall weighted:", recall_score(y_true, y_pred, average='weighted', zero_division=0))
    print("F1 weighted:", f1_score(y_true, y_pred, average='weighted', zero_division=0))

    print("Precision macro:", precision_score(y_true, y_pred, average='macro', zero_division=0))
    print("Recall macro:", recall_score(y_true, y_pred, average='macro', zero_division=0))
    print("F1 macro:", f1_score(y_true, y_pred, average='macro', zero_division=0))

    print("\nClassification Report:")
    print(classification_report(y_true, y_pred, zero_division=0))

    print("\nConfusion Matrix:")
    print(confusion_matrix(y_true, y_pred))


# =============================
# 1. Base model đánh giá trên test SMOTE
# =============================
# Vì rf_model là base model, nên X_test_smote cần scale bằng scaler của base model.
# Nếu scaler base của bạn tên là scaler thì dùng như dưới.

X_test_smote_for_base_compare = scaler.transform(X_test_smote)

X_test_smote_for_base_compare = pd.DataFrame(
    X_test_smote_for_base_compare,
    columns=X_test_smote.columns,
    index=X_test_smote.index
)

y_pred_base_compare_smote = rf_model.predict(X_test_smote_for_base_compare)

evaluate_model_compare(
    "BASE MODEL - Đánh giá trên tập test SMOTE",
    y_test_smote,
    y_pred_base_compare_smote
)


# =============================
# 2. SMOTE model đánh giá trên test SMOTE
# =============================
# Nếu SMOTE model được train bằng X_train_smote_scaled,
# thì dùng trực tiếp X_test_smote_scaled.

y_pred_smote_compare = rf_smote_model.predict(X_test_smote_scaled)

evaluate_model_compare(
    "SMOTE MODEL - Đánh giá trên cùng tập test SMOTE",
    y_test_smote,
    y_pred_smote_compare
)


BASE MODEL - Đánh giá trên tập test SMOTE
Accuracy: 0.9433198380566802
Balanced Accuracy: 0.9433115348008965
Precision weighted: 0.9473995417447095
Recall weighted: 0.9433198380566802
F1 weighted: 0.9433760667987616
Precision macro: 0.9473526609500477
Recall macro: 0.9433115348008965
F1 macro: 0.943346927836136

Classification Report:
              precision    recall  f1-score   support

           1       0.88      1.00      0.94       329
           2       0.97      0.88      0.92       329
           3       0.99      0.95      0.97       330

    accuracy                           0.94       988
   macro avg       0.95      0.94      0.94       988
weighted avg       0.95      0.94      0.94       988


Confusion Matrix:
[[328   1   0]
 [ 37 290   2]
 [  7   9 314]]

SMOTE MODEL - Đánh giá trên cùng tập test SMOTE
Accuracy: 0.9817813765182186
Balanced Accuracy: 0.981769058364803
Precision weighted: 0.9818178562587375
Recall weighted: 0.9817813765182186
F1 weighted: 0.98179512249

In [ ]:
# =============================
# 3. Bảng so sánh tổng hợp Base vs SMOTE trên test SMOTE
# =============================
compare_base_smote_on_smote_test = pd.DataFrame({
    "Model": [
        "Base model on SMOTE test",
        "SMOTE model on SMOTE test"
    ],
    "Accuracy": [
        accuracy_score(y_test_smote, y_pred_base_compare_smote),
        accuracy_score(y_test_smote, y_pred_smote_compare)
    ],
    "Balanced Accuracy": [
        balanced_accuracy_score(y_test_smote, y_pred_base_compare_smote),
        balanced_accuracy_score(y_test_smote, y_pred_smote_compare)
    ],
    "Precision Weighted": [
        precision_score(y_test_smote, y_pred_base_compare_smote, average='weighted', zero_division=0),
        precision_score(y_test_smote, y_pred_smote_compare, average='weighted', zero_division=0)
    ],
    "Recall Weighted": [
        recall_score(y_test_smote, y_pred_base_compare_smote, average='weighted', zero_division=0),
        recall_score(y_test_smote, y_pred_smote_compare, average='weighted', zero_division=0)
    ],
    "F1 Weighted": [
        f1_score(y_test_smote, y_pred_base_compare_smote, average='weighted', zero_division=0),
        f1_score(y_test_smote, y_pred_smote_compare, average='weighted', zero_division=0)
    ],
    "Precision Macro": [
        precision_score(y_test_smote, y_pred_base_compare_smote, average='macro', zero_division=0),
        precision_score(y_test_smote, y_pred_smote_compare, average='macro', zero_division=0)
    ],
    "Recall Macro": [
        recall_score(y_test_smote, y_pred_base_compare_smote, average='macro', zero_division=0),
        recall_score(y_test_smote, y_pred_smote_compare, average='macro', zero_division=0)
    ],
    "F1 Macro": [
        f1_score(y_test_smote, y_pred_base_compare_smote, average='macro', zero_division=0),
        f1_score(y_test_smote, y_pred_smote_compare, average='macro', zero_division=0)
    ]
})

display(compare_base_smote_on_smote_test)

,Model,Accuracy,Balanced Accuracy,Precision Weighted,Recall Weighted,F1 Weighted,Precision Macro,Recall Macro,F1 Macro
0,Base model on SMOTE test,0.943320,0.943312,0.947400,0.943320,0.943376,0.947353,0.943312,0.943347
1,SMOTE model on SMOTE test,0.981781,0.981769,0.981818,0.981781,0.981795,0.981803,0.981769,0.981781


Mang base ra dự đoán với GAN test

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)
import pandas as pd

# =============================
# Hàm đánh giá mô hình
# =============================
def evaluate_model_compare(model_name, y_true, y_pred):
    print("\n" + "="*60)
    print(model_name)
    print("="*60)

    print("Accuracy:", accuracy_score(y_true, y_pred))
    print("Balanced Accuracy:", balanced_accuracy_score(y_true, y_pred))

    print("Precision weighted:", precision_score(y_true, y_pred, average='weighted', zero_division=0))
    print("Recall weighted:", recall_score(y_true, y_pred, average='weighted', zero_division=0))
    print("F1 weighted:", f1_score(y_true, y_pred, average='weighted', zero_division=0))

    print("Precision macro:", precision_score(y_true, y_pred, average='macro', zero_division=0))
    print("Recall macro:", recall_score(y_true, y_pred, average='macro', zero_division=0))
    print("F1 macro:", f1_score(y_true, y_pred, average='macro', zero_division=0))

    print("\nClassification Report:")
    print(classification_report(y_true, y_pred, zero_division=0))

    print("\nConfusion Matrix:")
    print(confusion_matrix(y_true, y_pred))


# =============================
# 1. Base model đánh giá trên test GAN
# =============================
# Vì rf_model là base model, nên X_test_gan cần scale bằng scaler của base model.
# Nếu scaler base của bạn tên là scaler thì dùng như dưới.

X_test_gan_for_base_compare = scaler.transform(X_test_gan)

X_test_gan_for_base_compare = pd.DataFrame(
    X_test_gan_for_base_compare,
    columns=X_test_gan.columns,
    index=X_test_gan.index
)

y_pred_base_compare_gan_test = rf_model.predict(X_test_gan_for_base_compare)

evaluate_model_compare(
    "BASE MODEL - Đánh giá trên tập test GAN",
    y_test_gan,
    y_pred_base_compare_gan_test
)


# =============================
# 2. GAN model đánh giá trên test GAN
# =============================
# Nếu GAN model được train bằng X_train_gan_scaled,
# thì dùng trực tiếp X_test_gan_scaled.

y_pred_gan_on_gan_test = rf_gan_model.predict(X_test_gan_scaled)

evaluate_model_compare(
    "GAN MODEL - Đánh giá trên cùng tập test GAN",
    y_test_gan,
    y_pred_gan_on_gan_test
)


BASE MODEL - Đánh giá trên tập test GAN
Accuracy: 0.5425101214574899
Balanced Accuracy: 0.5426882809861534
Precision weighted: 0.6402259268773188
Recall weighted: 0.5425101214574899
F1 weighted: 0.5069970666071375
Precision macro: 0.6399092804957237
Recall macro: 0.5426882809861534
F1 macro: 0.5069742262679345

Classification Report:
              precision    recall  f1-score   support

           1       0.48      1.00      0.65       329
           2       0.49      0.26      0.34       329
           3       0.95      0.37      0.53       330

    accuracy                           0.54       988
   macro avg       0.64      0.54      0.51       988
weighted avg       0.64      0.54      0.51       988


Confusion Matrix:
[[328   1   0]
 [236  87   6]
 [118  91 121]]

GAN MODEL - Đánh giá trên cùng tập test GAN
Accuracy: 0.9564777327935222
Balanced Accuracy: 0.9564889011697523
Precision weighted: 0.9563074528634465
Recall weighted: 0.9564777327935222
F1 weighted: 0.956321933936730

In [ ]:
# =============================
# 3. Bảng so sánh tổng hợp Base vs GAN trên test GAN
# =============================
compare_base_gan_on_gan_test = pd.DataFrame({
    "Model": [
        "Base model on GAN test",
        "GAN model on GAN test"
    ],
    "Accuracy": [
        accuracy_score(y_test_gan, y_pred_base_compare_gan_test),
        accuracy_score(y_test_gan, y_pred_gan_on_gan_test)
    ],
    "Balanced Accuracy": [
        balanced_accuracy_score(y_test_gan, y_pred_base_compare_gan_test),
        balanced_accuracy_score(y_test_gan, y_pred_gan_on_gan_test)
    ],
    "Precision Weighted": [
        precision_score(y_test_gan, y_pred_base_compare_gan_test, average='weighted', zero_division=0),
        precision_score(y_test_gan, y_pred_gan_on_gan_test, average='weighted', zero_division=0)
    ],
    "Recall Weighted": [
        recall_score(y_test_gan, y_pred_base_compare_gan_test, average='weighted', zero_division=0),
        recall_score(y_test_gan, y_pred_gan_on_gan_test, average='weighted', zero_division=0)
    ],
    "F1 Weighted": [
        f1_score(y_test_gan, y_pred_base_compare_gan_test, average='weighted', zero_division=0),
        f1_score(y_test_gan, y_pred_gan_on_gan_test, average='weighted', zero_division=0)
    ],
    "Precision Macro": [
        precision_score(y_test_gan, y_pred_base_compare_gan_test, average='macro', zero_division=0),
        precision_score(y_test_gan, y_pred_gan_on_gan_test, average='macro', zero_division=0)
    ],
    "Recall Macro": [
        recall_score(y_test_gan, y_pred_base_compare_gan_test, average='macro', zero_division=0),
        recall_score(y_test_gan, y_pred_gan_on_gan_test, average='macro', zero_division=0)
    ],
    "F1 Macro": [
        f1_score(y_test_gan, y_pred_base_compare_gan_test, average='macro', zero_division=0),
        f1_score(y_test_gan, y_pred_gan_on_gan_test, average='macro', zero_division=0)
    ]
})

display(compare_base_gan_on_gan_test.round(4))

,Model,Accuracy,Balanced Accuracy,Precision Weighted,Recall Weighted,F1 Weighted,Precision Macro,Recall Macro,F1 Macro
0,Base model on GAN test,0.5425,0.5427,0.6402,0.5425,0.5070,0.6399,0.5427,0.5070
1,GAN model on GAN test,0.9565,0.9565,0.9563,0.9565,0.9563,0.9563,0.9565,0.9563
